# Ticha HTR — label smoothing and the cosine schedule, one change at a time

In `ticha_trocr_finetuning.ipynb` the first run (linear schedule, no label smoothing, 5 epochs) and the second (label smoothing 0.1, cosine schedule, up to 12 epochs) differ in three things at once, so the second run's gain (validation CER 18.7% → 17.9%) can't be split between them. This notebook trains the missing runs with **everything else identical to the second run**:

| Run | Label smoothing | Schedule | Epochs | Status |
|---|---|---|---|---|
| A | none | linear | up to 12 | trained here |
| B | 0.1 | linear | up to 12 | trained here |
| C | none | cosine | up to 12 | trained here |
| D | 0.1 | cosine | up to 12 | `trocr-base-handwritten-finetuned-smooth-cosine` (existing) |
| first run | none | linear | 5 | `trocr-base-handwritten-finetuned` (existing, for reference) |
| E | 0.1 | linear | up to 12, **with augmentation** | trained here (added after B beat D on validation) |
| D + augmentation | 0.1 | cosine | up to 12, with augmentation | `trocr-base-handwritten-finetuned-smooth-cosine-aug` (existing, `e2e-aug`) |

- **A vs D** shows what smoothing and cosine together add once the epochs are equal; **A vs the first run** shows what the longer training alone adds. **B** and **C** split smoothing from the schedule.
- **Same everything else:** the model setup, collator and arguments are copied from the "Experiment: label smoothing, cosine schedule and augmentation" section of `ticha_trocr_finetuning.ipynb` (AdamW, lr 5e-5, weight decay 0.01, 5% warmup, batch 8, seed 42, bf16), with early stopping on validation CER (patience 2) and the best epoch kept.
- **Same data as A's comparison runs:** the original train lines of dataset **v1.1** (5,305 lines after cleaning), the same as in v1.0, so they include Th687, as for D and the first run. Its duplicate Za687 is not in the v1.1 test split.
- **E vs D + augmentation** asks whether B's edge over D (validation 17.4% vs 17.9%) survives augmentation. E uses the augmented train lines exactly as `e2e-aug` did (v1.1, the same as v1.0, cleaned like the originals, original + augmented shuffled with seed 42, 26,525 lines), so the schedule is the only difference. E also scores a fixed sample of 710 original train lines every epoch (as many as validation), so its training CER per epoch sits next to its validation CER. Both are also scored on the degraded test copy (`test_degraded`, the same lines).
- **Same evaluation for all runs:** validation and test decoded with 2 beams (as during training) and start token `0`, and a paired bootstrap over whole documents against D.

**Before running:** train the three runs of [`ticha_trocr_finetuning.ipynb`](ticha_trocr_finetuning.ipynb) (D, the first run and D + augmentation); this notebook reads them from `$TICHA_RUNS_DIR` (default `~/ticha-runs`).

Run on the VM (kernel **Python (ticha-trocr)**): about 1 hour per run. A trained run is reloaded on a rerun; its epoch checkpoints are deleted once its best model is saved (the disk has room for one run's checkpoints at a time).

In [ ]:
import os
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')
import gc
import json
import re
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import concatenate_datasets, load_dataset
from IPython.display import display
from jiwer import cer, process_characters, wer
from transformers import (EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments, TrOCRProcessor,
                          VisionEncoderDecoderModel)
from transformers.models.vision_encoder_decoder.modeling_vision_encoder_decoder import shift_tokens_right

RUNS = Path(os.environ.get('TICHA_RUNS_DIR', '~/ticha-runs')).expanduser()
BASE = RUNS / 'trocr-base-handwritten-finetuned'
OUT = RUNS / 'ablation-smoothing-schedule'
MODEL_ID = 'microsoft/trocr-base-handwritten'
DATASET_REVISION = 'v1.1'  # train as in v1.0 (with Th687); Za687, its duplicate, is not in test
SMOKE = os.environ.get('TICHA_SMOKE') == '1'  # quick end-to-end check in a scratch folder
if SMOKE:
    OUT = RUNS / 'smoke-ablation'
OUT.mkdir(parents=True, exist_ok=True)

ABLATION_RUNS = {
    'A no smoothing, linear': dict(label_smoothing_factor=0.0, lr_scheduler_type='linear'),
    'B smoothing, linear': dict(label_smoothing_factor=0.1, lr_scheduler_type='linear'),
    'C no smoothing, cosine': dict(label_smoothing_factor=0.0, lr_scheduler_type='cosine'),
    'E smoothing, linear + augmentation': dict(label_smoothing_factor=0.1, lr_scheduler_type='linear', augmented=True),
}
EXISTING_RUNS = {
    'D smoothing, cosine (existing)': RUNS / 'trocr-base-handwritten-finetuned-smooth-cosine',
    'first run: linear, 5 epochs (existing)': BASE,
    'D + augmentation (existing)': RUNS / 'trocr-base-handwritten-finetuned-smooth-cosine-aug',
}
AUGMENTED_RUNS = ['E smoothing, linear + augmentation', 'D + augmentation (existing)']
if SMOKE:  # disk is tight: the smoke check trains only the new run
    ABLATION_RUNS = {k: v for k, v in ABLATION_RUNS.items() if k.startswith('E ')}
START = 0
EVAL_BEAMS = 2
DECODE_BATCH = 16
SEED = 42

if not torch.cuda.is_available():
    raise RuntimeError('No GPU: run this notebook on the VM.')
DEVICE = torch.device('cuda')
USE_BF16 = torch.cuda.is_bf16_supported()
BATCH_SIZE = 8 if torch.cuda.get_device_properties(0).total_memory / 1024**3 >= 20 else 2
print('GPU:', torch.cuda.get_device_name(0), '| batch size:', BATCH_SIZE, '| bf16:', USE_BF16, '| outputs:', OUT)

## Data: dataset v1.1, cleaned as in the training notebook

The same `clean_text` as `ticha_trocr_finetuning.ipynb` (uncertain readings kept as text in train, dropped from validation and test). The text is cleaned in memory and lines are kept by index, so no copy of the images is written to the datasets cache.

In [ ]:
BRACKET_RE = re.compile(r'\[([^\[\]]*)\]|\(([^()]*)\)')
NOTE_RE = re.compile(
    r'[?…]|^\.+$|\.\.|\s|,|^x+$|^word$'
    r'|cortado|roto|torn|mancha|borrado|firmas|ilegible',
    re.IGNORECASE)
DROP_HASH = True
GUESS_RE = re.compile(r'\((?!word\?)([^()?]*[^\W\d_][^()?]*)\?+\)')


# Same rule as ticha.core.htr.clean_text, written out so the notebook runs without the package.
def clean_text(text, keep_guesses=False):
    """Return the diplomatic text, or None when the line should be dropped."""
    text = str(text or '')
    if keep_guesses:
        text = GUESS_RE.sub(lambda m: m.group(1), text)
    notes = [square or round for square, round in BRACKET_RE.findall(text)]
    if any(NOTE_RE.search(note.strip()) for note in notes if not note.strip().startswith('=')):
        return None
    text = BRACKET_RE.sub('', text)
    if re.search(r'[\[\]()]', text) or (DROP_HASH and '#' in text):
        return None
    return re.sub(r'\s+', ' ', text).strip() or None


ds = load_dataset('maclenn77/ticha-htr-lines', 'original', revision=DATASET_REVISION)
for name in ds:
    before = len(ds[name])
    cleaned = [clean_text(t, keep_guesses=name == 'train') for t in ds[name]['text']]
    ds[name] = ds[name].remove_columns('text').add_column('text', cleaned)
    ds[name] = ds[name].select([i for i, t in enumerate(cleaned) if t is not None])
    print(f'{name}: kept {len(ds[name]):,} / {before:,} lines')
assert SMOKE or len(ds['train']) == 5305, 'not the train set D and the first run were trained on'


def clean_in_memory(split, keep_guesses):
    cleaned = [clean_text(t, keep_guesses=keep_guesses) for t in split['text']]
    split = split.remove_columns('text').add_column('text', cleaned)
    return split.select([i for i, t in enumerate(cleaned) if t is not None])


# Augmented lines (run E): the train copies, cleaned like the original train lines, plus the degraded test copy.
aug = load_dataset('maclenn77/ticha-htr-lines', 'augmented', revision=DATASET_REVISION)
shared = ['image', 'text', 'doc_id']
aug_train = clean_in_memory(aug['train'], keep_guesses=True).select_columns(shared)
if aug_train.features != ds['train'].select_columns(shared).features:
    aug_train = aug_train.cast(ds['train'].select_columns(shared).features)
train_aug = concatenate_datasets([ds['train'].select_columns(shared), aug_train]).shuffle(seed=SEED)
print(f'augmented train: {len(aug_train):,} lines -> {len(train_aug):,} training lines with the originals')
assert SMOKE or len(train_aug) == 26525, 'not the train set e2e-aug was trained on'
test_ids = set(ds['test']['id'])
degraded = clean_in_memory(aug['test_degraded'], keep_guesses=False)
ds['test_degraded'] = degraded.select([i for i, s in enumerate(degraded['source_id']) if s in test_ids])
print(f"test_degraded: {len(ds['test_degraded']):,} lines")
# Run E's training CER per epoch: a fixed sample of original (not augmented) train lines, as many as validation.
train_sample = ds['train'].shuffle(seed=SEED).select(range(len(ds['validation'])))

if SMOKE:
    ds['train'], ds['validation'], ds['test'] = (ds['train'].select(range(64)), ds['validation'].select(range(16)),
                                                 ds['test'].select(range(16)))
    ds['test_degraded'], train_aug = ds['test_degraded'].select(range(16)), train_aug.select(range(64))
    train_sample = ds['train'].select(range(16))
meta = {name: ds[name].remove_columns('image').to_pandas() for name in ['validation', 'test', 'test_degraded']}

## Model setup and training (copied from the experiment section of `ticha_trocr_finetuning.ipynb`)

In [ ]:
processor = TrOCRProcessor.from_pretrained(MODEL_ID)
tokenizer = processor.tokenizer
token_lengths = [len(tokenizer(t, add_special_tokens=True).input_ids) for t in ds['train']['text']]
MAX_TARGET_LENGTH = min(256, max(64, int(np.percentile(token_lengths, 99)) + 8))
print('MAX_TARGET_LENGTH:', MAX_TARGET_LENGTH)


class TrOCRDataCollator:
    def __init__(self, processor, max_target_length):
        self.processor = processor
        self.max_target_length = max_target_length

    def __call__(self, batch):
        images = [item['image'].convert('RGB') for item in batch]
        pixel_values = self.processor(images=images, return_tensors='pt').pixel_values
        texts = [str(item['text'] or '').strip() for item in batch]
        labels = self.processor.tokenizer(texts, padding='max_length', max_length=self.max_target_length,
                                          truncation=True, return_tensors='pt').input_ids
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {'pixel_values': pixel_values, 'labels': labels}


class TrOCRSmoothingCollator(TrOCRDataCollator):
    """With label smoothing the Trainer removes `labels` before calling the model, so the model can no longer
    build its decoder inputs from them: add them to the batch. Used for every run, so all four see the same batches."""

    def __call__(self, batch):
        out = super().__call__(batch)
        out['decoder_input_ids'] = shift_tokens_right(
            out['labels'], self.processor.tokenizer.pad_token_id, self.processor.tokenizer.cls_token_id)
        return out


def fresh_model():
    m = VisionEncoderDecoderModel.from_pretrained(MODEL_ID)
    m.config.decoder_start_token_id = processor.tokenizer.cls_token_id
    m.config.pad_token_id = processor.tokenizer.pad_token_id
    m.config.eos_token_id = processor.tokenizer.sep_token_id
    m.config.vocab_size = m.config.decoder.vocab_size
    m.generation_config.max_length = MAX_TARGET_LENGTH
    m.generation_config.num_beams = 2
    m.generation_config.early_stopping = True
    return m


def compute_metrics(eval_pred):
    pred_ids = eval_pred.predictions[0] if isinstance(eval_pred.predictions, tuple) else eval_pred.predictions
    label_ids = np.array(eval_pred.label_ids, copy=True)
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_text = [t.strip() for t in processor.batch_decode(pred_ids, skip_special_tokens=True)]
    ref_text = [t.strip() for t in processor.batch_decode(label_ids, skip_special_tokens=True)]
    return {'cer': cer(ref_text, pred_text), 'wer': wer(ref_text, pred_text)}


def run_dir(name):
    return OUT / re.sub(r'[^A-Za-z0-9]+', '-', name).strip('-').lower()


for name, changes in ABLATION_RUNS.items():
    changes = dict(changes)
    augmented = changes.pop('augmented', False)
    train_set = train_aug if augmented else ds['train']
    # E also scores the train sample each epoch; with two eval sets the metrics are named eval_validation_cer and
    # eval_train_sample_cer, and the best epoch is still chosen on validation.
    eval_set = {'validation': ds['validation'], 'train_sample': train_sample} if augmented else ds['validation']
    best_metric = 'validation_cer' if augmented else 'cer'
    out_dir = run_dir(name)
    if (out_dir / 'model.safetensors').exists():
        print(f'{name}: using the saved model in {out_dir}')
        continue
    args = Seq2SeqTrainingArguments(
        output_dir=str(out_dir), per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=8 // BATCH_SIZE, optim='adamw_torch', learning_rate=5e-5, weight_decay=0.01,
        warmup_ratio=0.05, num_train_epochs=12, eval_strategy='epoch', save_strategy='epoch',
        logging_strategy='steps', logging_steps=100, save_total_limit=2, load_best_model_at_end=True,
        metric_for_best_model=best_metric, greater_is_better=False, predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH, generation_num_beams=2, fp16=not USE_BF16, bf16=USE_BF16,
        remove_unused_columns=False, dataloader_num_workers=4, report_to='none', seed=SEED,
        **changes, **({'max_steps': 10} if SMOKE else {}))
    trainer = Seq2SeqTrainer(model=fresh_model(), args=args, train_dataset=train_set, eval_dataset=eval_set,
                             data_collator=TrOCRSmoothingCollator(processor, MAX_TARGET_LENGTH),
                             compute_metrics=compute_metrics,
                             callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    print(f'\n=== {name}: {changes}, {len(train_set):,} training lines -> {out_dir}', flush=True)
    t0 = time.time()
    trainer.train()
    trainer.save_model(str(out_dir))  # the best checkpoint (load_best_model_at_end)
    processor.save_pretrained(str(out_dir))
    trainer.save_state()
    print(f'{name}: trained in {(time.time() - t0) / 60:.0f} min', flush=True)
    del trainer
    gc.collect()
    torch.cuda.empty_cache()
    for checkpoint in out_dir.glob('checkpoint-*'):  # the best model is saved above; keep room for the next run
        shutil.rmtree(checkpoint)

In [ ]:
def training_summary(model_dir):
    state = json.loads((Path(model_dir) / 'trainer_state.json').read_text())
    key = 'eval_cer' if any('eval_cer' in h for h in state['log_history']) else 'eval_validation_cer'
    evals = [h for h in state['log_history'] if key in h]
    best = min(evals, key=lambda h: h[key])
    return {'best epoch': int(round(best['epoch'])), 'epochs run': int(round(evals[-1]['epoch'])),
            'validation CER (training log)': best[key], 'per epoch': [round(h[key], 4) for h in evals]}


all_runs = {name: run_dir(name) for name in ABLATION_RUNS}
all_runs.update(EXISTING_RUNS)
curves = pd.DataFrame({name: training_summary(path) for name, path in all_runs.items()}).T
print('Validation CER per epoch, from each run\'s training log (2 beams):')
display(curves)


# Run E: training CER (710 original train lines) next to validation CER, per epoch.
e_state = json.loads((run_dir('E smoothing, linear + augmentation') / 'trainer_state.json').read_text())
e_curve = {}
for h in e_state['log_history']:
    for split in ['validation', 'train_sample']:
        if f'eval_{split}_cer' in h:
            e_curve.setdefault(int(round(h['epoch'])), {})[f'{split} CER'] = h[f'eval_{split}_cer']
e_curve = pd.DataFrame(e_curve).T.rename_axis('epoch')
print('Run E, CER per epoch (2 beams): training sample vs validation')
display(e_curve.round(4))
e_curve.to_csv(OUT / 'e_train_vs_validation_cer.csv')

## The same evaluation for all five runs

Validation and test decoded with 2 beams and start token `0`. Saved predictions are scored on the lines of the loaded splits only (those made before v1.1 can include Za687).

In [ ]:
references = {s: dict(zip(meta[s]['id'], meta[s]['text'])) for s in meta}
languages = {s: dict(zip(meta[s]['id'], meta[s]['language'])) for s in meta}
# Only the lines of the loaded splits: predictions saved before v1.1 can include Za687.
keep = {split: set(ids).__contains__ for split, ids in references.items()}


def configure(m):
    for cfg in (m.config, m.generation_config):
        cfg.decoder_start_token_id = START
        cfg.pad_token_id, cfg.eos_token_id = tokenizer.pad_token_id, tokenizer.sep_token_id
    m.generation_config.max_length = MAX_TARGET_LENGTH
    m.generation_config.early_stopping = True
    return m


@torch.inference_mode()
def predict(model_dir, split_name):
    path = OUT / 'predictions' / f"{re.sub(r'[^A-Za-z0-9]+', '-', str(model_dir.name)).strip('-')}__{split_name}.csv"
    if path.exists():
        df = pd.read_csv(path, keep_default_na=False)
        return dict(zip(df['id'], df['prediction']))
    model = configure(VisionEncoderDecoderModel.from_pretrained(model_dir)).to(DEVICE).eval()
    split, out = ds[split_name], {}
    ids = split['id']
    for start in range(0, len(split), DECODE_BATCH):
        rows = split.select(range(start, min(start + DECODE_BATCH, len(split))))
        pixel_values = processor(images=[im.convert('RGB') for im in rows['image']], return_tensors='pt').pixel_values
        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=USE_BF16):
            seqs = model.generate(pixel_values=pixel_values.to(DEVICE), num_beams=EVAL_BEAMS,
                                  max_length=MAX_TARGET_LENGTH, early_stopping=True)
        for k, text in enumerate(tokenizer.batch_decode(seqs, skip_special_tokens=True)):
            out[ids[start + k]] = text.strip()
    path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({'id': list(out), 'reference': [references[split_name][i] for i in out],
                  'prediction': list(out.values())}).to_csv(path, index=False)
    del model
    torch.cuda.empty_cache()
    return out


def evaluate(split, preds):
    ids = [i for i in preds if keep[split](i)]
    refs, hyps = [references[split][i] for i in ids], [preds[i] or ' ' for i in ids]
    row = {f'{split} CER': cer(refs, hyps), f'{split} WER': wer(refs, hyps)}
    if split == 'test':
        row['test lines'] = len(ids)
        for lang in sorted({languages[split][i] for i in ids}):
            idx = [n for n, i in enumerate(ids) if languages[split][i] == lang]
            row[f'test CER {lang}'] = cer([refs[n] for n in idx], [hyps[n] for n in idx])
    return row


def paired_bootstrap(base, new, n=5000):
    ids = sorted(i for i in base if keep['test'](i))
    def errors(preds):
        out = []
        for i in ids:
            o = process_characters(references['test'][i], preds[i] or ' ')
            out.append(o.substitutions + o.deletions + o.insertions)
        return np.array(out)
    e_base, e_new = errors(base), errors(new)
    lengths = np.array([len(references['test'][i]) for i in ids])
    docs = np.array([i.split('_')[0] for i in ids])
    groups = [np.where(docs == d)[0] for d in np.unique(docs)]
    rng = np.random.default_rng(SEED)
    diffs = []
    for _ in range(n):
        s = np.concatenate([groups[g] for g in rng.integers(len(groups), size=len(groups))])
        diffs.append((e_new[s].sum() - e_base[s].sum()) / lengths[s].sum())
    diffs = np.array(diffs)
    return (e_new.sum() - e_base.sum()) / lengths.sum(), np.percentile(diffs, 2.5), np.percentile(diffs, 97.5), np.mean(diffs < 0)


predictions = {name: {s: predict(Path(path), s) for s in ['validation', 'test']} for name, path in all_runs.items()}
table = pd.DataFrame({name: {**{k: v for k, v in training_summary(path).items() if k != 'per epoch'},
                             **evaluate('validation', predictions[name]['validation']),
                             **evaluate('test', predictions[name]['test'])}
                      for name, path in all_runs.items()}).T
print('All runs, decoded the same way (2 beams, start token 0; test without Za687)')
display(table)
table.to_csv(OUT / 'ablation_results.csv')

reference_run = 'D smoothing, cosine (existing)'
for name in all_runs:
    if name != reference_run:
        d, lo, hi, share = paired_bootstrap(predictions[reference_run]['test'], predictions[name]['test'])
        print(f'{name} minus D, test CER: {d:+.4f} [{lo:+.4f}, {hi:+.4f}], better than D in {share:.1%} of resamples')
if 'A no smoothing, linear' in predictions:  # not in the smoke check, which trains only E
    d, lo, hi, share = paired_bootstrap(predictions['first run: linear, 5 epochs (existing)']['test'],
                                        predictions['A no smoothing, linear']['test'])
    print(f'A minus the first run (longer training only), test CER: {d:+.4f} [{lo:+.4f}, {hi:+.4f}], '
          f'better in {share:.1%} of resamples')

# With augmentation: does the linear schedule (E) still beat cosine (D + augmentation)?
e_name, daug_name = AUGMENTED_RUNS
d, lo, hi, share = paired_bootstrap(predictions[daug_name]['test'], predictions[e_name]['test'])
print(f'E minus D + augmentation, test CER: {d:+.4f} [{lo:+.4f}, {hi:+.4f}], better in {share:.1%} of resamples')
degraded_table = pd.DataFrame({name: {**evaluate('test', predictions[name]['test']),
                                      **evaluate('test_degraded', predict(Path(all_runs[name]), 'test_degraded'))}
                               for name in AUGMENTED_RUNS}).T[['test CER', 'test_degraded CER', 'test WER', 'test_degraded WER']]
degraded_table['degradation (CER points)'] = 100 * (degraded_table['test_degraded CER'] - degraded_table['test CER'])
print('Real vs degraded scans (test without Za687):')
display(degraded_table)
degraded_table.to_csv(OUT / 'augmented_runs_degraded.csv')

## How to read the results

- **First run → A:** the effect of training up to 12 epochs with early stopping, with the first run's settings.
- **A → D:** what label smoothing and the cosine schedule add together, at equal epochs. **B** and **C** show each alone.
- **E vs D + augmentation:** the schedule with augmentation. Only a validation gain counts for choosing; test and its interval confirm it.
- These are single runs with one seed. Differences of a few tenths of a point may come from randomness; a difference only counts if its bootstrap interval excludes zero, and even then a second seed is the real check.